# Data quality selections (also known as cuts)

To improve the quality of our data we usually apply quality selections (cuts) that help us to remove either background events or poorly reconstructed outlier signals. In general it is advisable to have the same rigorous tracking of settings and inputs for quality cuts as for normal data types. This is especially important when quality selections are shared among many users. 

Therefore, strax provides a dedicated `strax.CutPlugin` to develop individual quality cuts and a `CutList` plugin to combine multiple cuts into a single selection, e.g. used in a specific analysis.

# How to define a cut in strax

Defining a cut via a `CutPlugin` works almost identically to regular plugins. First, let us set up our context again:

In [ ]:
from pathlib import Path

import numpy as np

import strax
import abstrax 

SHORT_RUN = '012882'

# Read-only example data shipped with abstrax (path relative to the tutorials folder)
TEST_DATA = Path('../test_data').resolve()
# Writable folder for everything we produce. Safe to delete.
OUTPUT_FOLDER = Path('./strax_data').resolve()

st = abstrax.contexts.abstrax_context(output_folder=OUTPUT_FOLDER)
st.storage.append(strax.DataDirectory(str(TEST_DATA), readonly=True))

Now let us write two simple cuts. In contrast to a `strax.Plugin`, a `CutPlugin` only has to return a simple boolean array. Strax then automatically adds the time and endtime columns for you. In addition, each `CutPlugin` comes with two new properties called `cut_name` and `cut_description` that help to identify cuts.

In [ ]:
class ValidS1S2Pair(strax.CutPlugin):
    """
    Cut which tests if an event has both an S1 and an S2 peak.
    """

    depends_on = "event_basics"
    provides = "cut_event_is_valid"
    cut_name = "cut_event_is_valid"
    cut_description = "Removes all events that have only either a valid S1 or a valid S2."
    __version__ = "0.0.1"

    def cut_by(self, events):
        _has_s1_and_s2 = (
            ~np.isnan(events["s1_area"])
            & ~np.isnan(events["s2_area"]) 
        )
        return _has_s1_and_s2

Let us register this cut to see its outcome:

In [14]:
st.register(ValidS1S2Pair)

__main__.ValidS1S2Pair

Since the cut is of the same data kind as events, we can load both together and check the result:

In [15]:
event_basics = st.get_array(SHORT_RUN, ('event_basics', 'cut_event_is_valid'))
event_basics[['s1_area', 'cut_event_is_valid']]

array([(  7.735708,  True), (192.6994  ,  True), (       nan, False),
       (138.3644  ,  True), (       nan, False), (       nan, False),
       (       nan, False), (208.17053 ,  True), (       nan, False),
       (       nan, False), (151.58757 ,  True), (       nan, False),
       (       nan, False), (208.64944 ,  True), (       nan, False),
       (       nan, False), ( 64.234436,  True), (101.95314 ,  True),
       (       nan, False), ( 10.294443,  True)],
      dtype={'names': ['s1_area', 'cut_event_is_valid'], 'formats': ['<f4', '?'], 'offsets': [97, 16], 'titles': ['Main S1 area, uncorrected [PE]', 'Cut by valid_s1_s2_pair'], 'itemsize': 433})

Let's define a second cut so that we can define a cut list:

In [ ]:
class S2Threshold(strax.CutPlugin):
    """
    Applies area based threshold to events.
    """

    depends_on = "event_basics"
    provides = "cut_s2_area"
    cut_name = "cut_s2_area"
    cut_description = "Applies upper and lower boundary on the S2 area."
    __version__ = "0.0.1"

    s2_area_lower_bound = strax.URLConfig(
            default=1000,
            type=float,
            track=True,
            help="Lower boundary on the S2 area.",
    )

    s2_area_upper_bound = strax.URLConfig(
            default=10**6,
            type=float,
            track=True,
            help="Upper boundary on the S2 area.",
    )

    def cut_by(self, events):
        _s2_area_is_in_bounds = (
            (events["s2_area"] >= self.s2_area_lower_bound)
            & (events["s2_area"] < self.s2_area_upper_bound) 
        )
        return _s2_area_is_in_bounds

In [22]:
st.register(S2Threshold)

__main__.S2Threshold

In [23]:
event_basics = st.get_array(SHORT_RUN, ('event_basics', 'cut_event_is_valid', 'cut_s2_area'))
event_basics[['s2_area', 'cut_s2_area']]

array([(  881.78705, False), ( 5119.381  ,  True), ( 3268.7812 ,  True),
       ( 5976.1704 ,  True), ( 1201.9509 ,  True), ( 2115.8025 ,  True),
       ( 1154.2352 ,  True), (12587.818  ,  True), (  931.02606, False),
       (  514.87897, False), ( 1705.1791 ,  True), ( 1023.79675,  True),
       ( 2440.4836 ,  True), ( 7303.1514 ,  True), ( 1552.4724 ,  True),
       (  845.1104 , False), ( 1463.4869 ,  True), ( 1276.1274 ,  True),
       ( 1139.1361 ,  True), (  580.23883, False)],
      dtype={'names': ['s2_area', 'cut_s2_area'], 'formats': ['<f4', '?'], 'offsets': [246, 17], 'titles': ['Main S2 area, uncorrected [PE]', 'Applies upper and lower boundary on the S2 area.'], 'itemsize': 434})

# Cut lists

With more than one cut in place we can start defining cut lists. Cut lists are a convenient way to accumulate and share multiple cuts for the same analysis. They can be defined in the following way:

In [59]:
class MinimalCuts(strax.CutList):
    """A minimal selection that returns only valid 
    triggering events.
    """

    provides = "cuts_minimal"
    accumulated_cuts_string = "cuts_minimal"
    cut_description = "Decides if an event is a valid triggering event."
    __version__ = "0.0.1"
    cuts = (ValidS1S2Pair, S2Threshold,)

When registering a cut list we can use `st.register_cut_list`, which registers not only the `CutList`, but also all cuts it includes. To test this, let us first make a new context and see if `S2Threshold` is registered:

In [60]:
st = abstrax.contexts.abstrax_context(output_folder=OUTPUT_FOLDER)
st.storage.append(strax.DataDirectory(str(TEST_DATA), readonly=True))

try: 
    st.get_single_plugin(SHORT_RUN, 'cut_s2_area')
except KeyError as error:
    print('S2Threshold is not among the registered plugins')

S2Threshold is not among the registered plugins


Now register the cut list...

In [61]:
st.register_cut_list(MinimalCuts)

... and check again

In [62]:
try: 
    st.get_single_plugin(SHORT_RUN, 'cut_s2_area')
except KeyError as error:
    print('S2Threshold is not among the registered plugins')

We can also simply load events with the cut list:

In [63]:
events = st.get_array(SHORT_RUN, ('event_basics', 'cuts_minimal'))

And the loaded data does not only contain the combined cut, but also the individual subcuts:

In [65]:
events[['cut_event_is_valid', 'cut_s2_area', 'cuts_minimal']]

array([( True, False, False), ( True,  True,  True),
       (False,  True, False), ( True,  True,  True),
       (False,  True, False), (False,  True, False),
       (False,  True, False), ( True,  True,  True),
       (False, False, False), (False, False, False),
       ( True,  True,  True), (False,  True, False),
       (False,  True, False), ( True,  True,  True),
       (False,  True, False), (False, False, False),
       ( True,  True,  True), ( True,  True,  True),
       (False,  True, False), ( True, False, False)],
      dtype={'names': ['cut_event_is_valid', 'cut_s2_area', 'cuts_minimal'], 'formats': ['?', '?', '?'], 'offsets': [16, 17, 18], 'titles': ['Removes all events that have only either a valid S1 or a valid S2.', 'Applies upper and lower boundary on the S2 area.', 'Boolean AND of all cuts in cuts_minimal'], 'itemsize': 435})

Since cuts are usually trivial to compute, they are not stored by default: 

In [66]:
st.is_stored(SHORT_RUN, 'cut_s2_area')

False

If you have a computationally heavy cut you may want to save the result to disk explicitly:

In [67]:
st.make(SHORT_RUN, 'cut_s2_area', save='cut_s2_area')
st.is_stored(SHORT_RUN, 'cut_s2_area')

True